In [3]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

# 1. Load Data
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("✅ Loaded Data.")
except FileNotFoundError:
    print("❌ Files not found.")
    raise

print("--- STEP 1: ADVANCED FEATURE ENGINEERING (Your Best Setup) ---")

def create_features(df):
    df = df.copy()
    
    # 1. Total Activity Score
    activity_cols = ['hobby_engagement_level', 'physical_activity_index', 
                     'creative_expression_index', 'altruism_score']
    df['total_activity'] = df[activity_cols].sum(axis=1)
    
    # 2. Cluster A Detector
    df['support_guidance_combo'] = df['support_environment_score'] * (df['external_guidance_usage'] + 1)
    
    # 3. Focus Efficiency
    df['focus_efficiency'] = df['focus_intensity'] / (df['consistency_score'] + 1)
    
    # 4. Consistency Gap
    df['consistency_gap'] = 30 - df['consistency_score']
    
    return df

# Apply engineering
train_df = create_features(train_df)
test_df = create_features(test_df)

# 2. Prepare X and y
X = train_df.drop(['participant_id', 'personality_cluster'], axis=1)
y = train_df['personality_cluster']

test_ids = test_df['participant_id']
X_test_submit = test_df.drop(['participant_id'], axis=1)

# 3. Encode Categorical Features
# We keep your original categorical list
cat_features = ['identity_code', 'cultural_background', 'age_group']

for col in cat_features:
    X[col] = X[col].astype(str)
    X_test_submit[col] = X_test_submit[col].astype(str)

# 4. Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 5. Split (Stratify is crucial here)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

print(f"--- STARTING OPTUNA TUNING ---")
print(f"Training Shape: {X_train.shape}")

# 6. Define Optuna Objective
def objective(trial):
    # Suggest hyperparameters range
    param = {
        "iterations": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        
        # Essential Fixed Params
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        "auto_class_weights": "Balanced", # <--- KEEPING THIS (Your winning factor)
        "verbose": False,
        "random_seed": 42,
        "allow_writing_files": False
    }

    model = CatBoostClassifier(**param)
    
    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50
    )
    
    # Calculate Macro F1
    preds = model.predict(X_val)
    score = f1_score(y_val, preds, average="macro")
    
    return score

# 7. Run Optimization
# Run 20 trials to find the best combo
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100) 

print("\n✅ BEST PARAMETERS FOUND:")
print(study.best_params)
print(f"🏆 BEST VALIDATION MACRO F1: {study.best_value:.4f}")

# 8. Train Final Model with Best Params
print("\n--- Retraining Final Model ---")
best_params = study.best_params
# Add the fixed params back in
best_params["iterations"] = 2500
best_params["loss_function"] = "MultiClass"
best_params["eval_metric"] = "TotalF1"
best_params["auto_class_weights"] = "Balanced"
best_params["random_seed"] = 42

final_model = CatBoostClassifier(**best_params)
final_model.fit(
    X_train, y_train, 
    cat_features=cat_features,
    eval_set=(X_val, y_val), 
    verbose=200, 
    early_stopping_rounds=100
)

# 9. Save Submission
print("\n--- Saving Submission ---")
predictions_encoded = final_model.predict(X_test_submit)
predictions_labels = le.inverse_transform(predictions_encoded.flatten().astype(int))

submission = pd.DataFrame({
    'participant_id': test_ids,
    'personality_cluster': predictions_labels
})
submission.to_csv('submission_optuna_advanced.csv', index=False)
print("✅ Saved to 'submission_optuna_advanced.csv'")

[I 2025-11-19 22:38:09,476] A new study created in memory with name: no-name-10e503e8-dc91-4100-b4f5-2e3c75e74506


✅ Loaded Data.
--- STEP 1: ADVANCED FEATURE ENGINEERING (Your Best Setup) ---
--- STARTING OPTUNA TUNING ---
Training Shape: (1530, 16)


[I 2025-11-19 22:38:14,149] Trial 0 finished with value: 0.5451875291795082 and parameters: {'learning_rate': 0.04301964240274236, 'depth': 8, 'l2_leaf_reg': 3.0013461100112124, 'random_strength': 4.6285080325442385, 'bagging_temperature': 0.502218376879004}. Best is trial 0 with value: 0.5451875291795082.
[I 2025-11-19 22:38:26,690] Trial 1 finished with value: 0.5769270999052771 and parameters: {'learning_rate': 0.05016114504870399, 'depth': 6, 'l2_leaf_reg': 4.284713334972143, 'random_strength': 2.0975803169725835, 'bagging_temperature': 0.8399099704353029}. Best is trial 1 with value: 0.5769270999052771.
[I 2025-11-19 22:39:00,872] Trial 2 finished with value: 0.6191666382818992 and parameters: {'learning_rate': 0.06834540482843753, 'depth': 7, 'l2_leaf_reg': 3.6958002924416777, 'random_strength': 8.081563893103612, 'bagging_temperature': 0.3708637168043144}. Best is trial 2 with value: 0.6191666382818992.
[I 2025-11-19 22:39:16,245] Trial 3 finished with value: 0.5311364029979513 


✅ BEST PARAMETERS FOUND:
{'learning_rate': 0.09297972233363988, 'depth': 7, 'l2_leaf_reg': 4.21544666081472, 'random_strength': 0.38820330177846285, 'bagging_temperature': 0.12172592720791837}
🏆 BEST VALIDATION MACRO F1: 0.6463

--- Retraining Final Model ---
0:	learn: 0.5885985	test: 0.4629203	best: 0.4629203 (0)	total: 141ms	remaining: 5m 51s
200:	learn: 0.9098032	test: 0.6042151	best: 0.6278827 (136)	total: 22.2s	remaining: 4m 14s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6278827446
bestIteration = 136

Shrink model to first 137 iterations.

--- Saving Submission ---
✅ Saved to 'submission_optuna_advanced.csv'
